# Phase-1 Document Loading
### Purpose: Extract text from PDFs and TXT files
### Author: Ram Prashanth Rao G
### Date: Dec 10, 2025

## This Notebook covers
1. How to load PDFs using PyPDF
2. How to load text files
3. How to build universal document loader
4. How to preserve metadata for RAG

In [2]:
# Import libraries for document loading

from pathlib import Path ## Required to construct file paths that work on any OS
from PyPDF2 import PdfReader ## Extracts text from the pdf, could use upgraded tools like pdfplumber

# Print versions to verify installation
print("Libraries imported successfully!")
print(f"PyPDF2 version: {PdfReader.__module__}")


Libraries imported successfully!
PyPDF2 version: PyPDF2._reader


In [4]:
### Loading the AI Ethics Policy PDF

pdf_path = Path("../data/documents/sap/Global_AI_Ethics_Policy.pdf")

## Create a PDF reader object
reader = PdfReader(pdf_path)

## Get no.of pages
num_pages = len(reader.pages)
print(f"PDF loaded: {pdf_path.name}")
print(f"Number of pages: {num_pages}")

## Extract text from the first page only (to see what it looks like)
first_page = reader.pages[0]
first_page_text = first_page.extract_text()

## Print first 500 chars
print("\n ---First 500 chars---")
print(first_page_text[:500])



PDF loaded: Global_AI_Ethics_Policy.pdf
Number of pages: 13

 ---First 500 chars---
 
 
                                                                                                                           Version 2.0 , August  01, 2024  
Document Cla ssification: PUBLIC  
The online version of this documents  is the officially released version.  
  Any copies or print -outs are not controlled  
 
PUBLIC  
Global Artificial Intelligence (AI) Ethics Policy  
  



In [11]:
### Reading the text completely from the pdf

full_text = ""

## loop through the 13 pages we've in the doc
for page in reader.pages:
    full_text += page.extract_text()

print(f"Total length of the pdf file is: {len(full_text)}")

Total length of the pdf file is: 32407


In [13]:
### Loading txt files --> txt files has only chars so its easier to load than PDf
### PDF has bin format, pages, fonts, images

txt_path = Path("../data/documents/general/remote_work_policy.txt")

# Read the text file
with open(txt_path, 'r', encoding='utf-8') as file:
    txt_content = file.read()

# Print the info
print(f"TXT loaded: {txt_path.name}")
print(f"Total characters: {len(txt_content)}")
print(f"\n ----First 500 characters----")
print(txt_content[:500])


TXT loaded: remote_work_policy.txt
Total characters: 1751

 ----First 500 characters----
# Remote Work Policy

Effective Date: January 1, 2024

## Eligibility
Employees must complete a 6-month probationary period and receive manager approval. Role must be suitable for remote work with demonstrated ability to work independently.

## Work Schedule
Standard business hours apply (9 AM - 5 PM local time). Core collaboration hours are 10 AM - 3 PM when all team members must be available. Flexible scheduling available with manager approval.

## Equipment and Technology
Company provides lap


In [15]:
### Define a Universal loader function ###

def load_document(file_path):
    """ Load a document and extract its text.

    Args: file_path: Path object pointing to the file

    Returns: dict with 'text' and 'metadata' keys

    """

    # Convert to Path object 
    file_path = Path(file_path)

    # Get file extension
    ext = file_path.suffix.lower()

    # Initialize the variables
    text = ""
    doc_type = ""

    ## conditions
    if ext == '.pdf':
        doc_type = "pdf"
        reader = PdfReader(file_path)
        for page in reader.pages:
            text += page.extract_text()

    elif ext == '.txt':
        doc_type = "txt"
        with open(file_path, 'r', encoding='utf-8') as file:
            text = file.read()

    else:
        raise ValueError(f"Unsupported file type: {ext}")

     # Return text and metadata
    return {
        'text': text,
        'metadata': {
            'source': file_path.name,
            'type': doc_type,
            'path': str(file_path),
            'chars': len(text)
        }
    }


        
    

In [17]:
# Test 1: Load a PDF
print("=== Testing PDF Loading ===")
pdf_result = load_document("../data/documents/sap/Global_AI_Ethics_Policy.pdf")
print(f"Source: {pdf_result['metadata']['source']}")
print(f"Type: {pdf_result['metadata']['type']}")
print(f"Characters: {pdf_result['metadata']['chars']}")
print(f"First 200 chars: {pdf_result['text'][:200]}")

print("\n=== Testing TXT Loading ===")
# Test 2: Load a TXT file
txt_result = load_document("../data/documents/general/remote_work_policy.txt")
print(f"Source: {txt_result['metadata']['source']}")
print(f"Type: {txt_result['metadata']['type']}")
print(f"Characters: {txt_result['metadata']['chars']}")
print(f"First 200 chars: {txt_result['text'][:200]}")


print("\n✅ Universal loader works for both formats!")


=== Testing PDF Loading ===
Source: Global_AI_Ethics_Policy.pdf
Type: pdf
Characters: 32407
First 200 chars:  
 
                                                                                                                           Version 2.0 , August  01, 2024  
Document Cla ssification: PUBLIC  
The o

=== Testing TXT Loading ===
Source: remote_work_policy.txt
Type: txt
Characters: 1751
First 200 chars: # Remote Work Policy

Effective Date: January 1, 2024

## Eligibility
Employees must complete a 6-month probationary period and receive manager approval. Role must be suitable for remote work with dem

✅ Universal loader works for both formats!


In [18]:
### Loading all documents

folder = Path("../data/documents")
txt_files = folder.rglob("*.txt")
pdf_files = folder.rglob("*.pdf")

# Store values in a list
documents = []
for file in txt_files:
    result = load_document(file)
    documents.append(result)

for file in pdf_files:
    result = load_document(file)
    documents.append(result)


print(f"Loaded {len(documents)} documents")
print("\nDocument Summary:")
for doc in documents:
    print(f"  - {doc['metadata']['source']}: {doc['metadata']['chars']} chars")

Loaded 8 documents

Document Summary:
  - remote_work_policy.txt: 1751 chars
  - git_workflow.txt: 1545 chars
  - deployment_checklist.txt: 1879 chars
  - api_development_standards.txt: 1957 chars
  - security_guidelines.txt: 1682 chars
  - Global_CoE_BizConduct.pdf: 63639 chars
  - SAP_Partner_CoC.pdf: 26251 chars
  - Global_AI_Ethics_Policy.pdf: 32407 chars
